# GSM8K zero-shot transfer of the HH prefix scorer

Attach the original `karimvafin/cd-hh-scorer` Kaggle dataset and enable GPU and internet. Run with `N_PROMPTS = 16` for a smoke test, then use the full official test split for the comparison.

All methods use the same zero-shot prompt, frozen generator, and HH sampling configuration. The answer format is `#### <number>`. The primary endpoint is exact numeric match (missing or malformed answers count as incorrect). CD vs direct sampling measures the effect of deploying CD. CD vs random block selection measures the effect of scoring with the same block length and branching factor. Actual generated-token totals may differ because selected answers can finish at different lengths.

The output cap is 512 tokens for all methods. This is longer than the HH training cap. Report EOS and answer extraction rates to help interpret any differences. Choose the acceptable loss in percentage points before inspecting full results.


In [ ]:
import json
import re
from decimal import Decimal
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoModelForTokenClassification, AutoTokenizer

N_PROMPTS = None  # None = full official test set
SEED = 0
MAX_NEW_TOKENS = 512
M, K = 32, 4  # Chosen from the HH reproduction for its time/quality trade-off
SHARD_SIZE = 16
GEN_SEQS = 64
SCORER_BATCH = 32
NONINFERIORITY_MARGIN_PP = 2.0  # Example margin; fix before inspecting full results
GEN_DEVICE = "cuda:0"
AUX_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else GEN_DEVICE
DTYPE = torch.float16
SCORER_DIR = None  # Set explicitly when more than one scorer dataset is attached


In [ ]:
assert torch.cuda.is_available(), "Enable a GPU accelerator."
if SCORER_DIR is None:
    configs = list(Path("/kaggle/input").rglob("scorer_config.json"))
    assert len(configs) == 1, f"Attach the original HH scorer only, or set SCORER_DIR. Found: {configs}"
    SCORER = configs[0].parent
else:
    SCORER = Path(SCORER_DIR)
scorer_cfg = json.loads((SCORER / "scorer_config.json").read_text())
rollout_cfg = scorer_cfg["rollouts_cfg"]
assert not scorer_cfg["train_meta"]["debug"] and not rollout_cfg["debug"]
assert rollout_cfg["base_model"] == "Qwen/Qwen2.5-0.5B-Instruct"
assert rollout_cfg["gen_kw"].get("temperature", 1.0) == 1.0
assert rollout_cfg["gen_kw"].get("top_p", 1.0) == 1.0
GEN_KW = {key: value for key, value in rollout_cfg["gen_kw"].items() if key != "max_new_tokens"}

tok = AutoTokenizer.from_pretrained(rollout_cfg["base_model"])
tok.padding_side = "left"
policy = AutoModelForCausalLM.from_pretrained(rollout_cfg["base_model"], dtype=DTYPE).to(GEN_DEVICE).eval()
eos = policy.generation_config.eos_token_id
EOS_IDS = set(eos if isinstance(eos, list) else [eos])

scorer = AutoModelForTokenClassification.from_pretrained(SCORER, dtype=DTYPE).to(AUX_DEVICE).eval()
scorer.config.pad_token_id = tok.pad_token_id

def cut_at_eos(ids):
    for i, t in enumerate(ids):
        if t in EOS_IDS:
            return ids[: i + 1], True
    return ids, False

@torch.no_grad()
def sample(seqs, n, max_new):
    # n сэмплов π_ref на каждую последовательность; ответ i*n + k относится к seqs[i]
    out = []
    per_call = max(1, GEN_SEQS // n)
    for b in range(0, len(seqs), per_call):
        batch = tok.pad({"input_ids": seqs[b:b + per_call]}, return_tensors="pt").to(GEN_DEVICE)
        g = policy.generate(**batch, **GEN_KW, max_new_tokens=max_new, num_return_sequences=n, pad_token_id=tok.pad_token_id)
        out += [cut_at_eos(o) for o in g[:, batch["input_ids"].shape[1]:].tolist()]
    return out

@torch.no_grad()
def values(seqs):
    # V(x, y^t) на последнем токене каждой последовательности (промпт + префикс ответа)
    res = np.empty(len(seqs))
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    for k in range(0, len(order), SCORER_BATCH):
        idx = order[k:k + SCORER_BATCH]
        L = max(len(seqs[i]) for i in idx)
        ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):
            ids[r, :len(seqs[i])] = torch.tensor(seqs[i]); att[r, :len(seqs[i])] = 1
        v = scorer(input_ids=ids.to(AUX_DEVICE), attention_mask=att.to(AUX_DEVICE)).logits[..., 0].float()
        last = att.sum(1).to(AUX_DEVICE) - 1
        res[idx] = v[torch.arange(len(idx), device=AUX_DEVICE), last].cpu().numpy()
    return res



In [ ]:
NUMBER = r"[+-]?(?:[0-9]{1,3}(?:,[0-9]{3})+|[0-9]+)(?:\.[0-9]+)?"
ANSWER = re.compile(rf"^[ \t]*####[ \t]*({NUMBER})[ \t]*\.?[ \t]*$", re.MULTILINE)

def extract_answer(text):
    matches = ANSWER.findall(text)
    return Decimal(matches[-1].replace(",", "")) if matches else None

dataset = load_dataset("openai/gsm8k", "main", split="test")
indices = np.arange(len(dataset))
if N_PROMPTS is not None:
    indices = np.sort(np.random.default_rng(SEED).choice(indices, N_PROMPTS, replace=False))
test = []
for pid in indices:
    item = dataset[int(pid)]
    gold = extract_answer(item["answer"])
    assert gold is not None, f"Unparseable reference for question {pid}"
    prompt = (
        item["question"]
        + "\nSolve the problem. Put your final numeric answer on its own line in the format #### <number>."
    )
    prompt_ids = tok.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True,
        add_generation_prompt=True, return_dict=False,
    )
    test.append({"pid": int(pid), "gold": gold, "prompt_ids": prompt_ids})


In [ ]:
def run_blocks(prompt_ids, M, K, score_candidates):
    n = len(prompt_ids)
    resp, fin, n_blocks, gen_tok = [[] for _ in range(n)], [False] * n, [0] * n, [0] * n
    done = [False] * n
    while not all(done):
        act = [i for i in range(n) if not done[i]]
        seqs = [prompt_ids[i] + resp[i] for i in act]
        max_new = min(M, max(MAX_NEW_TOKENS - len(resp[i]) for i in act))
        cands = sample(seqs, K, max_new)
        flat, meta = [], []
        for j, i in enumerate(act):
            rem = MAX_NEW_TOKENS - len(resp[i])
            for k in range(K):
                ids, f = cands[j * K + k]
                gen_tok[i] += len(ids)
                if len(ids) > rem:
                    ids, f = ids[:rem], False
                flat.append(seqs[j] + ids); meta.append((ids, f))
        v = values(flat) if score_candidates else None
        for j, i in enumerate(act):
            # The first of K IID candidates is a uniform selection without scoring.
            offset = int(np.argmax(v[j * K:(j + 1) * K])) if score_candidates else 0
            ids, f = meta[j * K + offset]
            resp[i] += ids; n_blocks[i] += 1; fin[i] = f
            done[i] = f or len(resp[i]) >= MAX_NEW_TOKENS
    return resp, fin, n_blocks, gen_tok


In [ ]:
rows = []
methods = ("direct", "random_blocks", "hh_cd")
for start in tqdm(range(0, len(test), SHARD_SIZE), desc="GSM8K batches"):
    batch = test[start:start + SHARD_SIZE]
    prompts = [item["prompt_ids"] for item in batch]
    for method in methods:
        torch.manual_seed(SEED * 100_003 + start)
        started = perf_counter()
        if method == "direct":
            outputs = sample(prompts, 1, MAX_NEW_TOKENS)
            responses, finished = zip(*outputs)
            generated = [len(ids) for ids in responses]
        else:
            responses, finished, _, generated = run_blocks(prompts, M, K, method == "hh_cd")
        seconds = (perf_counter() - started) / len(batch)
        for item, ids, eos, cost in zip(batch, responses, finished, generated):
            response = tok.decode(ids, skip_special_tokens=True).strip()
            answer = extract_answer(response)
            rows.append({
                "pid": item["pid"], "method": method, "gold": str(item["gold"]),
                "prediction": None if answer is None else str(answer),
                "correct": answer == item["gold"], "parsed": answer is not None,
                "finished": eos, "response_tokens": len(ids), "generated_tokens": cost,
                "seconds_per_prompt": seconds, "response": response,
            })

results = pd.DataFrame(rows)
assert len(results) == len(test) * len(methods)
output = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
tag = "full" if N_PROMPTS is None else f"n{N_PROMPTS}"
results.to_csv(output / f"gsm8k_hh_transfer_{tag}_predictions.csv", index=False)


In [ ]:
summary = results.groupby("method", sort=False).agg(
    n=("pid", "size"), accuracy=("correct", "mean"), parsed_rate=("parsed", "mean"),
    eos_rate=("finished", "mean"), mean_response_tokens=("response_tokens", "mean"),
    mean_generated_tokens=("generated_tokens", "mean"),
    seconds_per_prompt=("seconds_per_prompt", "mean"),
)
wide = results.pivot(index="pid", columns="method", values="correct").astype(int)
rng = np.random.default_rng(SEED)
draws = rng.integers(len(wide), size=(10_000, len(wide)))
comparisons = []
for baseline in ("direct", "random_blocks"):
    delta = 100 * (wide.hh_cd - wide[baseline]).to_numpy()
    boot = delta[draws].mean(axis=1)
    low, high = np.quantile(boot, [0.025, 0.975])
    comparisons.append({
        "baseline": baseline, "delta_pp": delta.mean(), "ci_low_pp": low,
        "ci_high_pp": high, "cd_only_correct": int((delta > 0).sum()),
        "baseline_only_correct": int((delta < 0).sum()),
        "noninferior_at_margin": low > -NONINFERIORITY_MARGIN_PP,
        "margin_pp": NONINFERIORITY_MARGIN_PP,
    })
paired = pd.DataFrame(comparisons)
summary.to_csv(output / f"gsm8k_hh_transfer_{tag}_summary.csv")
paired.to_csv(output / f"gsm8k_hh_transfer_{tag}_paired.csv", index=False)
display(summary, paired)
